In [1]:
'''
problem statement : TO predict PowerComsumption
--------------------------------------------------------------------------------------------------------------------------------------------------------------
workflow :

Input feature (X):
target columns(y):

-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------
step1: data_ingestion
step2: data_exploration [ descritive stats]
step3: data_preprocessing 
    1)  drop the duplicates
    2)  split the X and y  i.e target or input feature
    3)  split the train test split .. for  data leakege
    4)  Use the scaling ...... depend of data [ regression / categorical  columns of target columns]
    5) Use the SMOTE ( if applicable )

step4: model_building 
step5: model_evaluation
step6: model deployement
'''

# import manipulation labraries
import pandas as pd
import numpy as np

# import visualization labraries
import matplotlib.pyplot as plt
import seaborn as sns

# import labraries for preprocessing 

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler , StandardScaler , RobustScaler , LabelEncoder ,OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score , mean_absolute_error
from sklearn.compose import ColumnTransformer 
from sklearn.impute import SimpleImputer



# import labraries for model building

from sklearn.linear_model import LinearRegression , Ridge , Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor , AdaBoostRegressor ,GradientBoostingRegressor , ExtraTreesRegressor
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor

from collections import OrderedDict
import os
import sys
# step1: data ingstion
data_path = os.path.join("data","data.csv")
def data_ingestion():
    df = pd.read_csv(data_path)
    df.columns
    return df

# step2: data exploation [ descreative stas]
def data_exploration(df):
    numerical_col = df.select_dtypes(exclude='object').columns
    categorical_col = df.select_dtypes(include='object').columns

    Q1 = df[numerical_col].quantile(0.25)
    Q3 = df[numerical_col].quantile(0.75)
    IQR = Q3 - Q1

    LW = Q1-1.5*IQR
    UW = Q3+1.5*IQR

    Outliers_count = df[(df[numerical_col] < LW) | (df[numerical_col] > UW)].shape[0]
    Outliers_percentage = Outliers_count / len(df) * 100

    stats=[]

    for i in numerical_col:
        numerical_stats = OrderedDict({
            'feature':i,
            'minimum':df[i].min(),
            'maximum':df[i].max(),
            'mean':df[i].mean(),
            'median':df[i].median(),
            'stander deviation':df[i].std(),
            'skewness':df[i].skew(),
            'kurtosis':df[i].kurt(),
            'Q1':Q1[i],
            'Q3':Q3[i],
            'IQR':IQR[i],
            
            
        })
        stats.append(numerical_stats)
        report = pd.DataFrame(stats)
        return report

# data_preprocessing
  # 1) drop the duplicates
  # 2) split the X and y i.e input feature and target columns
  # 3) split the train test split 
  # 4) use scaling ------> labelEncoder , OneHotEncoder
  # 5) use SMOTE   ( if applicable)
def Data_preprocessing(df):
    df=df.drop_duplicates()

   
    X = df.drop(columns=['PowerConsumption_Zone2'])
    y =df['PowerConsumption_Zone2']

    numerical_col = X.select_dtypes(exclude='object').columns
    categorical_col = X.select_dtypes(include='object').columns

    X_train, X_test, y_train, y_test = train_test_split(X,y,
                                                        test_size = 0.3,
                                                        random_state=3)


    num_pipeline= Pipeline(steps=[
        ("imputer",SimpleImputer(strategy="median")),
        ("scaler",RobustScaler())

    ])

    cat_col = Pipeline(steps=[
        ("imputer",SimpleImputer(strategy="most_frequent")),
        ("encoder",OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("num",num_pipeline,numerical_col),
        ("cat",cat_col,categorical_col)

    ])

    X_train = preprocessor.fit_transform(X_train)
    X_test = preprocessor.transform(X_test)
    return X_train , X_test,y_train,y_test

def model_building(X_train , X_test,y_train,y_test):
    model_building =({
        "DecisionTreeRegressor":DecisionTreeRegressor(max_depth=200, 
                                                    max_leaf_nodes=2,
                                                        min_samples_split=17),
        "RandomForestRegressor":RandomForestRegressor(n_estimators=200,
                                                    max_depth=100,),
        "AdaBoostRegressor":AdaBoostRegressor(n_estimators=300,
                                            random_state=5),
        "GradientBoostingRegressor":GradientBoostingRegressor(n_estimators=300,
                                                            max_depth=100, 
                                                            random_state=21),
        "ExtraTreesRegressor":ExtraTreesRegressor(n_estimators=300,
                                                max_depth=20,
                                                    random_state=15),
        "XGBRegressor":XGBRegressor(n_estimators=300 , 
                                    max_depth=6)

    })
    for model_name, model in model_building.items():
        model.fit(X_train,y_train)
        y_pred =model.predict(X_test)
        report= r2_score(y_test , y_pred)
        print(f"model name:{model_name} r2_score is :{report}")
    return report


In [2]:
def main():
    df = data_ingestion()
    X_train, X_test, y_train, y_test = Data_preprocessing(df)
    
    report = model_building(X_train, X_test, y_train, y_test)
    
    print(report)
main()

FileNotFoundError: [Errno 2] No such file or directory: 'data\\data.csv'

NameError: name 'df' is not defined